# CWT-Based Gaze Error Spectrogram Pipeline & Edge AI Engine

**Statistical Learning for AI Lab — VOG-MCI Detection System**

---

## Project Overview

This notebook implements a complete end-to-end pipeline for **binary classification of Mild Cognitive Impairment (MCI) vs. Healthy Controls (HC)** using Video-Oculography (VOG) saccade recordings.

The core premise is that MCI patients exhibit measurable degradation in oculomotor control — specifically, abnormal **gaze error dynamics** during visually-guided saccade tasks. By transforming the 1-D gaze error signal into the time-frequency domain via the Continuous Wavelet Transform (CWT), we obtain a 2-D scalogram that simultaneously encodes:
- **Reaction latency** (temporal axis): delayed saccade initiation
- **Micro-saccadic tremor** (frequency axis): high-frequency instability during fixation

### Pipeline Architecture

```
Raw VOG CSV
  │
  ▼
[Layer 1]  Event-Locked CWT Pipeline
           Trigger detection → Epoch extraction → Gaze error → Complex Morlet CWT
           Output: [2, 40, 100] tensor  (Real + Imag channels)
  │
  ├──────► [Layer 5]  XAI Visualizer
  │                   Group-mean dB scalograms → SPM-style Difference Map
  │
  ▼
[Layer 2]  VOG_CWT_Dataset
           Per-channel Z-score normalization, subject-ID tracking
  │
  ▼
[Layer 3]  EdgeCWTClassifier  (CNN–CBAM Hybrid)
           3× [DepthwiseSep-Conv → CBAM] → GAP → FC(64→32→2)
  │
  ▼
[Layer 4a] ModelTrainer
           Subject-level stratified split, Weighted CE Loss, CosineAnnealingLR
  │
  ▼
[Layer 4b] JetsonInferenceEngine
           Per-epoch inference → Soft-voting ensemble → MCI probability
```

### Dataset

| Group | Subjects (local) | CSV files | Label |
|-------|-----------------|-----------|-------|
| HC    | 14              | 116       | 0     |
| MCI   | 12              | 96        | 1     |
| MCI+  | 11              | 88        | 1     |
| **Total** | **37**      | **300**   | binary|

An additional cohort of comparable size (~37 subjects) is reserved for fine-tuning, yielding a total of approximately **74 subjects**.

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
import pywt
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

---

## Attention Module: CBAM (Convolutional Block Attention Module)

### Motivation

A plain CNN applies uniform weights across all channels and all spatial positions of a feature map. For CWT scalograms, however, the clinically informative signal is **sparse and localized** in time-frequency space — for example, MCI-related latency deficits concentrate around $t \in [0.2, 0.4]$ s post-stimulus, and tremor instability concentrates in $f \in [15, 30]$ Hz. CBAM provides a **data-driven gating mechanism** that suppresses uninformative regions and amplifies diagnostically relevant ones, directly aligning with the XAI objective of identifying time-frequency Regions of Interest (RoIs).

### Channel Attention — *What* to emphasize

Given a feature map $\mathbf{F} \in \mathbb{R}^{C \times H \times W}$, the channel attention map $\mathbf{M}_c \in \mathbb{R}^{C}$ is computed via a shared MLP over both average-pooled and max-pooled descriptors:

$$\mathbf{M}_c(\mathbf{F}) = \sigma\!\left(\mathrm{MLP}\bigl(\mathbf{F}^c_{\mathrm{avg}}\bigr) + \mathrm{MLP}\bigl(\mathbf{F}^c_{\mathrm{max}}\bigr)\right)$$

where $\mathbf{F}^c_{\mathrm{avg}} = \frac{1}{HW}\sum_{h,w} \mathbf{F}_{:,h,w}$ captures global statistics and $\mathbf{F}^c_{\mathrm{max}}$ captures salient activations. The shared MLP has a bottleneck of $\lfloor C/r \rfloor$ neurons ($r=4$ here).

### Spatial Attention — *Where* to look

The spatial attention map $\mathbf{M}_s \in \mathbb{R}^{1 \times H \times W}$ operates on channel-pooled representations:

$$\mathbf{M}_s(\mathbf{F}') = \sigma\!\left(f^{7\times7}\bigl([\mathbf{F}'^s_{\mathrm{avg}}\,;\,\mathbf{F}'^s_{\mathrm{max}}]\bigr)\right)$$

where $[\,;\,]$ denotes channel-wise concatenation and $f^{7\times7}$ is a $7\times7$ convolution. The $7\times7$ kernel spans a broad receptive field in the $(f, t)$ plane, appropriate for detecting distributed latency and frequency patterns.

The full CBAM refinement is applied sequentially: $\mathbf{F}'' = \mathbf{M}_s(\mathbf{F}') \otimes \mathbf{F}'$, where $\mathbf{F}' = \mathbf{M}_c(\mathbf{F}) \otimes \mathbf{F}$.

> **XAI connection:** The spatial attention weight matrix $\mathbf{M}_s$ at the final feature stage can be upsampled back to the original scalogram dimensions and overlaid as a learned saliency map — providing a direct, model-intrinsic explanation of *which time-frequency region drove the classification decision*.

In [ ]:
# =========================================================================
# CBAM: Convolutional Block Attention Module
# Channel attention (WHAT to focus on) + Spatial attention (WHERE to focus)
# Spatial attention aligns with XAI goal: highlight Time-Freq RoIs
# =========================================================================
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 2)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c = x.size(0), x.size(1)
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return self.sigmoid(avg + mx).view(b, c, 1, 1) * x


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg    = torch.mean(x, dim=1, keepdim=True)
        mx, _  = torch.max(x,  dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1))) * x


class CBAM(nn.Module):
    def __init__(self, channels, reduction=4, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        return self.sa(self.ca(x))

---

## Layer 1 — Event-Locked CWT Pipeline

### 1.1 Clinical Signal: Gaze Error During Saccades

A **saccade** is a rapid ballistic eye movement that re-foveates a newly appeared target. In MCI, documented oculomotor deficits include:
- **Increased latency** ($\Delta t \in [50, 200]$ ms): delayed initiation of the corrective saccade
- **Hypometria** (undershoot): the eye falls short of the target ($e > 0$)
- **Hypermetria** (overshoot): the eye overshoots the target ($e < 0$)
- **Increased post-saccadic oscillations**: high-frequency instability at fixation

The **signed gaze error** is defined as:

$$e(t) = \theta_{\mathrm{target}}(t) - \theta_{\mathrm{actual}}(t)$$

Note that the absolute value is deliberately *not* taken — preserving sign distinguishes hypometria ($e > 0$) from hypermetria ($e < 0$), which have different neural substrates and may carry distinct discriminative power for MCI.

### 1.2 Event-Locked Epoching

The stimulus onset time $t_k$ is detected as the first moment the target position changes:

$$t_k = \{t : \theta_{\mathrm{target}}(t) \neq \theta_{\mathrm{target}}(t - \Delta t)\}$$

Each epoch is extracted as a fixed window relative to $t_k$:

$$\mathcal{E}_k = e(t) \text{ for } t \in [t_k - 0.2\,\mathrm{s},\; t_k + 0.8\,\mathrm{s}]$$

**Baseline correction** removes the pre-stimulus DC offset to eliminate systematic fixation bias:

$$\tilde{e}_k(t) = e_k(t) - \underbrace{\frac{1}{N_{\mathrm{pre}}} \sum_{t < t_k} e_k(t)}_{\text{pre-stimulus mean}}$$

This is analogous to the demeaning step in ERP (Event-Related Potential) analysis and ensures the CWT captures *change from baseline* rather than absolute gaze position.

### 1.3 Continuous Wavelet Transform with Complex Morlet

The CWT of a signal $f(t)$ with respect to a mother wavelet $\psi$ is:

$$W_\psi[f](a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} f(t)\, \overline{\psi\!\left(\frac{t - b}{a}\right)} \, dt$$

where $a > 0$ is the **scale** (inversely proportional to frequency) and $b$ is the **time shift**. The frequency-scale relationship is:

$$f = \frac{f_c}{a \cdot \Delta t}$$

where $f_c$ is the center frequency of $\psi$. We use the **Complex Morlet wavelet** (`cmor5.0-1.0`):

$$\psi_{\mathrm{cmor}}(t) = \frac{1}{\sqrt{\pi B}} e^{2\pi i f_c t} \, e^{-t^2 / B}$$

with bandwidth parameter $B = 1.0$ and $f_c = 5.0$ Hz. This wavelet provides a favorable trade-off between **time resolution** (important for latency estimation) and **frequency resolution** (important for tremor characterization), governed by the Heisenberg-Gabor uncertainty principle: $\sigma_t \cdot \sigma_f \geq \frac{1}{4\pi}$.

**Superiority over STFT:** The STFT uses a fixed window length, yielding uniform time-frequency resolution. The CWT adaptively uses short windows at high frequencies (good time resolution for fast events) and long windows at low frequencies (good frequency resolution for slow oscillations) — critical for capturing both micro-saccadic tremor and reaction latency in the same scalogram.

### 1.4 Two-Channel Complex Output

Rather than discarding phase information via $|W_\psi[\tilde{e}]|^2$, the pipeline retains the full complex output as two separate channels:

$$\mathbf{Z}_k = \begin{bmatrix} \mathrm{Re}(W_\psi[\tilde{e}_k]) \\ \mathrm{Im}(W_\psi[\tilde{e}_k]) \end{bmatrix} \in \mathbb{R}^{2 \times F \times T}$$

The real part encodes cosine-phase components (aligned with the positive gaze error direction, i.e., hypometria), while the imaginary part encodes sine-phase components. Together they allow the network to reconstruct both magnitude and instantaneous phase, providing richer discriminative features than power alone.

In [ ]:
# =========================================================================
# [Layer 1] Data Engineering: Event-Locked CWT Pipeline
# Outputs 2-channel tensors [real, imag] to preserve directional phase
# (Hypometria = undershoot, Hypermetria = overshoot)
# data_store[group][subject_id][task][eye] = list of [2, freq_bins, time_bins]
# =========================================================================
class EventLockedCWTPipeline:
    def __init__(self, pre_stimulus_sec=0.2, post_stimulus_sec=0.8,
                 min_freq=1.0, max_freq=40.0, freq_bins=40,
                 target_time_bins=100, w_morlet=5.0):
        self.pre_sec          = pre_stimulus_sec
        self.post_sec         = post_stimulus_sec
        self.min_freq         = min_freq
        self.max_freq         = max_freq
        self.freq_bins        = freq_bins
        self.target_time_bins = target_time_bins
        self.w                = w_morlet
        self.frequencies      = np.linspace(min_freq, max_freq, freq_bins)

        self.target_tasks = {
            "Horizontal": ["Horizontal Saccade A", "Horizontal Saccade B",
                           "Horizontal Saccade B (anti)", "Horizontal Saccade R"],
            "Vertical":   ["Vertical Saccade A",   "Vertical Saccade B",
                           "Vertical Saccade B (anti)",   "Vertical Saccade R"],
        }
        # 4-level store: group -> subject_id -> task -> eye -> [tensors]
        self.data_store = defaultdict(
            lambda: defaultdict(
                lambda: defaultdict(
                    lambda: defaultdict(list)
                )
            )
        )

    # ------------------------------------------------------------------
    def _load_csv_safely(self, file_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(file_path, skipinitialspace=True)
            df.columns = [str(c).strip().lower() for c in df.columns]
            if any('lh' in c for c in df.columns):
                # Fast path: ensure numeric conversion before returning
                return df.apply(pd.to_numeric, errors='coerce').dropna(how='all').reset_index(drop=True)
        except Exception:
            pass

        for enc in ['utf-16', 'utf-16le', 'utf-8-sig', 'cp949']:
            try:
                with open(file_path, 'r', encoding=enc, errors='replace') as f:
                    lines = f.readlines()
                for i, line in enumerate(lines):
                    line_clean = line.replace('\x00', '').lower()
                    if 'lh' in line_clean and 'rh' in line_clean:
                        header_cols = [col.replace('\x00', '').strip().lower() for col in line.split(',')]
                        parsed = [
                            [v.strip() for v in l.replace('\x00', '').strip().split(',')]
                            for l in lines[i + 1:] if l.strip()
                        ]
                        df = pd.DataFrame(parsed, columns=header_cols)
                        return df.apply(pd.to_numeric, errors='coerce').dropna(how='all').reset_index(drop=True)
            except UnicodeError:
                continue
        raise ValueError(f"Headers missing or unreadable in {file_path.name}")

    # ------------------------------------------------------------------
    def _extract_epochs_and_compute_cwt(self, df, target_col, actual_col, current_fs):
        target_val  = df[target_col].fillna(0).values
        actual_val  = df[actual_col].fillna(0).values
        target_diff = np.diff(target_val, prepend=0)

        # All target transitions are valid saccade events (including return-to-zero)
        event_indices = np.where(target_diff != 0)[0]

        samples_pre  = int(self.pre_sec  * current_fs)
        samples_post = int(self.post_sec * current_fs)
        dt           = 1.0 / current_fs
        wavelet_name = f'cmor{self.w}-1.0'
        scales       = pywt.central_frequency(wavelet_name) / (self.frequencies * dt)

        valid_cwts = []
        for idx in event_indices:
            start_idx = idx - samples_pre
            end_idx   = idx + samples_post
            if start_idx < 0 or end_idx > len(df):  # > allows end_idx == len (valid slice)
                continue

            error_epoch    = target_val[start_idx:end_idx] - actual_val[start_idx:end_idx]
            baseline       = np.mean(error_epoch[:samples_pre])
            error_corrected = error_epoch - baseline

            cwtm, _ = pywt.cwt(error_corrected, scales, wavelet_name, sampling_period=dt)
            # Output: complex array [freq_bins, time_bins]
            # Preserve real + imaginary separately to retain directional phase
            time_zoom    = self.target_time_bins / cwtm.shape[1]
            real_resized = zoom(np.real(cwtm), (1.0, time_zoom), mode='nearest', order=1)
            imag_resized = zoom(np.imag(cwtm), (1.0, time_zoom), mode='nearest', order=1)

            valid_cwts.append(np.stack([real_resized, imag_resized], axis=0))  # [2, freq, time]
        return valid_cwts

    # ------------------------------------------------------------------
    def process_directory(self, base_dir: Path):
        csv_files = [f for f in base_dir.rglob('*.csv') if 'PD VOG' in f.name.upper()]
        for filepath in csv_files:
            clean_task = filepath.stem.replace("PD VOG -_", "").replace("PD VOG -", "").strip()

            axis_type = ("Horizontal" if "Horizontal" in clean_task
                         else "Vertical" if "Vertical" in clean_task else None)
            if not axis_type or clean_task not in self.target_tasks[axis_type]:
                continue

            # Traverse up from file to find HC/MCI group folder
            group = None
            cur = filepath.parent
            while cur != base_dir and cur != cur.parent:
                if cur.name.upper().startswith("HC"):  group = "HC"; break
                if cur.name.upper().startswith("MCI"): group = "MCI"; break
                cur = cur.parent
            if not group:
                for part in filepath.parts:
                    if part.upper().startswith("HC"):  group = "HC"; break
                    if part.upper().startswith("MCI"): group = "MCI"; break
            if not group:
                continue

            subject_id = filepath.parent.name  # timestamp folder = unique subject ID

            try:
                df        = self._load_csv_safely(filepath)
                is_anti   = "anti" in clean_task.lower()
                axis_char = 'h' if axis_type == "Horizontal" else 'v'

                time_col   = next((c for c in df.columns if 'time' in c or c == 't'), df.columns[0])
                time_val   = df[time_col].dropna().values
                current_fs = 1.0 / np.mean(np.diff(time_val)) if len(time_val) > 1 else 120.0

                target_col = next(
                    (c for c in df.columns if f'target{axis_char}' in c or f'target_{axis_char}' in c), None
                )
                if not target_col: continue
                if is_anti: df[target_col] = df[target_col] * -1

                for eye in ['l', 'r']:
                    actual_col = next((c for c in df.columns if c == f'{eye}{axis_char}'), None)
                    if not actual_col: continue

                    cwt_epochs = self._extract_epochs_and_compute_cwt(
                        df, target_col, actual_col, current_fs
                    )
                    if cwt_epochs:
                        eye_full = 'Left' if eye == 'l' else 'Right'
                        self.data_store[group][subject_id][clean_task][eye_full].extend(cwt_epochs)
            except Exception as e:
                print(f"[!] Skipped {filepath.name}: {e}")

---

## Layer 2 — PyTorch Dataset Bridge

### 2.1 Normalization

Each CWT tensor $\mathbf{Z}_k \in \mathbb{R}^{2 \times F \times T}$ is normalized independently per channel:

$$\hat{Z}_k^{(c)} = \frac{Z_k^{(c)} - \mu_k^{(c)}}{\sigma_k^{(c)} + \epsilon}, \quad c \in \{\mathrm{Re},\, \mathrm{Im}\}$$

where $\mu_k^{(c)}$ and $\sigma_k^{(c)}$ are the mean and standard deviation over all $(f, t)$ positions of channel $c$ in epoch $k$. Per-channel normalization is preferred over joint normalization to prevent the larger-magnitude real channel from dominating the imaginary channel, which carries complementary phase information.

The additive $\epsilon = 10^{-8}$ prevents division by zero for flat epochs (e.g., missing eye data).

### 2.2 Subject-ID Tracking for Leakage-Free Evaluation

Each sample in the dataset carries a `subject_id` identifier (the recording timestamp folder, unique per patient). This is critical: **epoch-level random splitting would allow multiple epochs from the same subject to appear in both the training and validation sets**, causing the model to learn subject-specific idiosyncrasies rather than generalizable MCI biomarkers — a form of data leakage that inflates validation accuracy.

The `subject_ids` list enables the `ModelTrainer` to perform a proper **subject-level stratified split**, where all epochs from a given subject appear exclusively in either the training set or the validation set — never both.

In [ ]:
# =========================================================================
# [Layer 2] PyTorch Dataset Bridge
# Normalizes each channel independently; tracks subject_id for leakage-free splits
# =========================================================================
class VOG_CWT_Dataset(Dataset):
    def __init__(self, data_store):
        self.X           = []
        self.y           = []
        self.subject_ids = []

        for group, subjects in data_store.items():
            label = 0 if group == "HC" else 1
            for subject_id, tasks in subjects.items():
                for task, eyes in tasks.items():
                    for eye, tensors in eyes.items():
                        for tensor in tensors:  # [2, freq_bins, time_bins]
                            # Z-score per channel to preserve relative real/imag magnitudes
                            norm_chs = []
                            for ch in range(tensor.shape[0]):
                                ch_data = tensor[ch]
                                norm_chs.append(
                                    (ch_data - np.mean(ch_data)) / (np.std(ch_data) + 1e-8)
                                )
                            self.X.append(np.stack(norm_chs, axis=0))
                            self.y.append(label)
                            self.subject_ids.append(subject_id)

        # Shape: [N, 2, freq_bins, time_bins] — no unsqueeze needed (already 2-channel)
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

---

## Layer 3 — EdgeCWTClassifier (CNN–CBAM Hybrid)

### Architecture Summary

The model is a lightweight CNN designed for real-time inference on the Jetson AGX Orin (Ampere GPU, 16–32 GB unified memory). It treats the 2-channel CWT scalogram as a 2D image with two input channels — analogous to how an RGB image has 3 channels — and applies a hierarchical feature extraction with attention gating.

| Stage | Operation | Output shape | Parameters |
|-------|-----------|-------------|------------|
| Input | — | `[B, 2, 40, 100]` | — |
| Block 1 | Conv(2→16, 3×3) + BN + ReLU + MaxPool(2×2) | `[B, 16, 20, 50]` | 304 |
| CBAM 1 | Channel(16) + Spatial(7×7) | `[B, 16, 20, 50]` | 386 |
| Block 2 | DW-Conv(16, 3×3) + PW-Conv(16→32, 1×1) + BN + ReLU + MaxPool | `[B, 32, 10, 25]` | 688 |
| CBAM 2 | Channel(32) + Spatial(7×7) | `[B, 32, 10, 25]` | 1,346 |
| Block 3 | DW-Conv(32, 3×3) + PW-Conv(32→64, 1×1) + BN + ReLU | `[B, 64, 10, 25]` | 2,368 |
| CBAM 3 | Channel(64) + Spatial(7×7) | `[B, 64, 10, 25]` | 4,994 |
| GAP | AdaptiveAvgPool(1×1) | `[B, 64]` | — |
| Head | Dropout(0.3) + FC(64→32) + ReLU + FC(32→2) | `[B, 2]` | 2,146 |

**Total trainable parameters: ~12,000** — intentionally minimal for a dataset of ~74 subjects.

### Design Rationale

**Depthwise-Separable Convolutions (Blocks 2 & 3):** Factorize a standard $k \times k$ convolution into a depthwise convolution (one filter per input channel) followed by a $1 \times 1$ pointwise convolution (channel mixing). For a $C_{in} \to C_{out}$ layer with kernel $k$, the parameter count reduces from $k^2 C_{in} C_{out}$ to $k^2 C_{in} + C_{in} C_{out}$, a reduction factor of $\approx 8\times$ at $k=3$.

**CBAM after Block 3, before GAP:** The Global Average Pooling collapses spatial dimensions to $1 \times 1$, destroying spatial information. Placing CBAM *before* GAP ensures the spatial attention operates on a feature map still carrying $(f, t)$ structure ($10 \times 25$ pixels), making the attention weights interpretable as a time-frequency saliency mask.

**Dropout(0.3):** Applied before the classifier head to regularize the final representation, critical given the small number of subjects.

In [ ]:
# =========================================================================
# [Layer 3] Edge AI Model: CNN-CBAM Hybrid (Jetson AGX Orin 최적화)
#
# Input: [B, 2, 40, 100]  (2-channel: real + imag CWT)
# Block 1: Standard conv     -> [B, 16, 20, 50]  + CBAM
# Block 2: DW-Sep conv       -> [B, 32, 10, 25]  + CBAM
# Block 3: DW-Sep conv       -> [B, 64, 10, 25]  + CBAM -> GAP -> [B, 64]
# Classifier: FC(64->32->2)
# =========================================================================
class EdgeCWTClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Block 1: standard conv, 2-channel input
        self.block1 = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.cbam1 = CBAM(16)

        # Block 2: MobileNet-style depthwise-separable (16 -> 32)
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16),  # depthwise
            nn.Conv2d(16, 32, kernel_size=1),                         # pointwise expand
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.cbam2 = CBAM(32)

        # Block 3: MobileNet-style depthwise-separable (32 -> 64)
        # GAP applied AFTER CBAM so spatial attention operates on full feature map
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32),  # depthwise
            nn.Conv2d(32, 64, kernel_size=1),                         # pointwise expand
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.cbam3 = CBAM(64)
        self.gap   = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = self.cbam1(self.block1(x))
        x = self.cbam2(self.block2(x))
        x = self.gap(self.cbam3(self.block3(x)))
        x = torch.flatten(x, 1)
        return self.classifier(x)

---

## Layer 4a — Model Trainer

### 4.1 Subject-Level Stratified Split

With $N \approx 74$ total subjects, preserving **statistical independence** between the training and validation cohorts is paramount. The split is performed at the **subject level**, not the epoch level:

1. Group all epoch indices by subject ID: $\mathcal{I}_s = \{i : \text{subject\_id}[i] = s\}$
2. Separately shuffle the HC subjects $\mathcal{S}_{\mathrm{HC}}$ and MCI subjects $\mathcal{S}_{\mathrm{MCI}}$
3. Assign the first $\lfloor 0.2 |\mathcal{S}_c| \rfloor$ subjects per class to the validation set
4. Map subject IDs back to epoch indices via $\mathcal{I}_s$

This guarantees that no information about a validation subject leaks into the training set through shared epochs — equivalent to the statistical requirement of independent and identically distributed (i.i.d.) test samples.

### 4.2 Weighted Cross-Entropy Loss

The dataset exhibits a class imbalance of approximately **1:1.6 (HC:MCI)**. Standard cross-entropy optimizes the average log-loss uniformly over samples, which on an imbalanced dataset biases the model toward the majority class (MCI). The weighted cross-entropy applies **inverse-frequency weighting**:

$$w_c = \frac{N}{K \cdot N_c}, \quad \mathcal{L}_{\mathrm{WCE}} = -\frac{1}{N}\sum_{i=1}^{N} w_{y_i} \log p(y_i \mid \mathbf{x}_i)$$

where $N$ is the total number of training epochs, $K=2$ is the number of classes, and $N_c$ is the count of class $c$. This ensures that a misclassification of the minority class (HC) incurs proportionally higher loss, encouraging the model to maintain sensitivity for both classes.

### 4.3 Optimization and Learning Rate Schedule

The optimizer is **AdamW** (Adam with decoupled weight decay), which applies L2 regularization directly to the weights rather than to the gradient update — correcting a flaw in the original Adam implementation and providing more reliable regularization:

$$\theta_{t+1} = \theta_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \alpha \lambda \theta_t$$

The learning rate follows a **Cosine Annealing** schedule:

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_0 - \alpha_{\min})\left(1 + \cos\frac{\pi t}{T_{\max}}\right)$$

with $\alpha_0 = 10^{-3}$ and $T_{\max} = $ `epochs`. This avoids the sharp learning rate drops of step-decay schedules and has been shown to improve generalization by exploring flat minima in the loss landscape.

### 4.4 Mixed-Precision Training (FP16)

On the Jetson AGX Orin's Ampere Tensor Cores, FP16 arithmetic runs at up to $2\times$ the throughput of FP32. PyTorch's `torch.amp.autocast` automatically casts eligible operations (convolutions, matrix multiplications) to FP16 while maintaining FP32 for numerically sensitive operations (batch norm, loss computation). The `GradScaler` multiplies the loss by a large factor before backpropagation to prevent FP16 gradient underflow, then unscales before the optimizer step.

In [ ]:
# =========================================================================
# [Layer 4a] Model Trainer
# - Subject-level stratified split (prevents patient data leakage)
# - Weighted CrossEntropyLoss (handles HC/MCI class imbalance)
# - CosineAnnealingLR scheduler
# - torch.amp API (PyTorch 2.x compatible)
# =========================================================================
class ModelTrainer:
    def __init__(self, model, device="cuda"):
        self.device    = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model     = model.to(self.device)
        self.use_amp   = self.device.type == "cuda"
        self.scaler    = torch.amp.GradScaler('cuda') if self.use_amp else None
        self.optimizer = optim.AdamW(self.model.parameters(), lr=1e-3, weight_decay=1e-4)

    def _subject_stratified_split(self, dataset, val_ratio=0.2, seed=42):
        random.seed(seed)
        subject_to_idx = defaultdict(list)
        for i, sid in enumerate(dataset.subject_ids):
            subject_to_idx[sid].append(i)

        hc_subjects  = [s for s in subject_to_idx if dataset.y[subject_to_idx[s][0]].item() == 0]
        mci_subjects = [s for s in subject_to_idx if dataset.y[subject_to_idx[s][0]].item() == 1]
        random.shuffle(hc_subjects)
        random.shuffle(mci_subjects)

        hc_val_n  = max(1, int(len(hc_subjects)  * val_ratio))
        mci_val_n = max(1, int(len(mci_subjects) * val_ratio))

        val_subjects   = set(hc_subjects[:hc_val_n]   + mci_subjects[:mci_val_n])
        train_subjects = set(hc_subjects[hc_val_n:]   + mci_subjects[mci_val_n:])

        train_idx = [i for i, s in enumerate(dataset.subject_ids) if s in train_subjects]
        val_idx   = [i for i, s in enumerate(dataset.subject_ids) if s in val_subjects]

        n_hc_train  = sum(1 for s in train_subjects if dataset.y[subject_to_idx[s][0]].item() == 0)
        n_mci_train = sum(1 for s in train_subjects if dataset.y[subject_to_idx[s][0]].item() == 1)
        print(f"[*] Train subjects: HC={n_hc_train}, MCI={n_mci_train} | "
              f"Val subjects: HC={hc_val_n}, MCI={mci_val_n}")

        return Subset(dataset, train_idx), Subset(dataset, val_idx)

    def train_model(self, dataset, epochs=20, batch_size=32):
        print(f"[*] 학습 시작 (디바이스: {self.device})")

        train_dataset, val_dataset = self._subject_stratified_split(dataset)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True)
        val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

        # Weighted loss: inverse-frequency weighting for HC/MCI imbalance
        label_counts  = Counter(dataset.y.tolist())
        total         = len(dataset)
        class_weights = torch.tensor(
            [total / (2 * label_counts[i]) for i in range(2)], dtype=torch.float32
        ).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f"[*] Class weights — HC: {class_weights[0]:.3f}, MCI: {class_weights[1]:.3f}")

        scheduler    = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=epochs)
        best_val_loss = float('inf')

        for epoch in range(epochs):
            # --- Training ---
            self.model.train()
            train_loss = 0.0; train_correct = 0; train_total = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    outputs = self.model(inputs)
                    loss    = criterion(outputs, labels)
                if self.use_amp:
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    loss.backward()
                    self.optimizer.step()
                train_loss    += loss.item() * inputs.size(0)
                _, predicted   = outputs.max(1)
                train_total   += labels.size(0)
                train_correct += predicted.eq(labels).sum().item()

            # --- Validation ---
            self.model.eval()
            val_loss = 0.0; val_correct = 0; val_total = 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(self.device), labels.to(self.device)
                    with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                        outputs = self.model(inputs)
                        loss    = criterion(outputs, labels)
                    val_loss    += loss.item() * inputs.size(0)
                    _, predicted = outputs.max(1)
                    val_total   += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

            train_acc    = 100. * train_correct / train_total
            val_acc      = 100. * val_correct   / val_total
            val_loss_avg = val_loss / val_total
            scheduler.step()

            print(f"Epoch [{epoch+1:02d}/{epochs}] "
                  f"| Train Acc: {train_acc:.2f}% "
                  f"| Val Acc: {val_acc:.2f}% "
                  f"| Val Loss: {val_loss_avg:.4f} "
                  f"| LR: {scheduler.get_last_lr()[0]:.2e}")

            if val_loss_avg < best_val_loss:
                best_val_loss = val_loss_avg
                torch.save(self.model.state_dict(), 'best_edge_cwt_model.pth')

        print("[+] 학습 완료. 'best_edge_cwt_model.pth' 저장됨.")

---

## Layer 4b — Jetson Inference Engine

### Epoch-Level Soft-Voting Ensemble

A single CSV file from a patient contains multiple saccade trials. Rather than classifying on a single epoch (which is noisy), the engine extracts $K$ CWT epochs from the file, runs inference on each, and aggregates predictions via **soft-voting** (averaging posterior probabilities):

$$\hat{p}(y = 1 \mid \mathcal{X}) = \frac{1}{K} \sum_{k=1}^{K} p(y = 1 \mid \mathbf{z}_k; \hat{\theta})$$

where $\mathbf{z}_k$ is the normalized CWT tensor for epoch $k$ and $\hat{\theta}$ are the trained model parameters. Under the assumption that individual epochs are conditionally independent given the patient's clinical state, this estimator has variance $\mathrm{Var}[\hat{p}] = \sigma^2_k / K$, improving robustness by a factor of $\sqrt{K}$ compared to single-epoch classification.

The final diagnosis threshold is $\hat{y} = \mathbf{1}[\hat{p}(y=1) > 0.5]$, with the MCI confidence score reported as a percentage.

In [ ]:
# =========================================================================
# [Layer 4b] Jetson Inference Engine
# =========================================================================
class JetsonInferenceEngine:
    def __init__(self, model_path, pipeline_config, device="cuda"):
        self.device   = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_amp  = self.device.type == "cuda"
        self.pipeline = EventLockedCWTPipeline(**pipeline_config)
        self.model    = EdgeCWTClassifier(num_classes=2)
        if os.path.exists(model_path):
            self.model.load_state_dict(
                torch.load(model_path, map_location=self.device, weights_only=True)
            )
        self.model.to(self.device).eval()
        self.classes = ["Healthy Control (HC)", "Mild Cognitive Impairment (MCI)"]

    def infer_csv(self, csv_filepath: Path):
        print(f"\n[*] 추론 시작: {csv_filepath.name}")
        df = self.pipeline._load_csv_safely(csv_filepath)

        axis_char  = 'h' if 'Horizontal' in csv_filepath.name else 'v'
        target_col = next(
            (c for c in df.columns if f'target{axis_char}' in c or f'target_{axis_char}' in c), None
        )
        actual_col = next((c for c in df.columns if c == f'l{axis_char}'), None)

        if not target_col or not actual_col:
            return "추론 불가: Target 또는 Eye 데이터를 찾을 수 없습니다."

        time_col   = next((c for c in df.columns if 'time' in c or c == 't'), df.columns[0])
        time_val   = df[time_col].dropna().values
        current_fs = 1.0 / np.mean(np.diff(time_val)) if len(time_val) > 1 else 120.0

        cwt_epochs = self.pipeline._extract_epochs_and_compute_cwt(
            df, target_col, actual_col, current_fs
        )
        if not cwt_epochs:
            return "추론 불가: 유효한 Saccade 이벤트가 없습니다."

        input_tensors = []
        for tensor in cwt_epochs:  # [2, freq, time]
            norm_chs = []
            for ch in range(tensor.shape[0]):
                ch_data = tensor[ch]
                norm_chs.append((ch_data - np.mean(ch_data)) / (np.std(ch_data) + 1e-8))
            input_tensors.append(np.stack(norm_chs, axis=0))

        inputs = torch.tensor(np.array(input_tensors), dtype=torch.float32).to(self.device)

        with torch.no_grad():
            with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                outputs       = self.model(inputs)
                probabilities = F.softmax(outputs, dim=1)

        mean_prob       = probabilities.mean(dim=0)
        predicted_class = torch.argmax(mean_prob).item()
        mci_confidence  = mean_prob[1].item() * 100

        print(f"[>] Saccade 이벤트 수: {len(cwt_epochs)}")
        print(f"[>] 앙상블 진단: {self.classes[predicted_class]}")
        print(f"[>] MCI 확률: {mci_confidence:.2f}%")
        return predicted_class, mci_confidence

---

## Layer 5 — XAI Visualizer: SPM-style Difference Maps

### Statistical Parametric Mapping in the Time-Frequency Domain

Statistical Parametric Mapping (SPM), originally developed for fMRI and EEG neuroimaging analysis, identifies voxels (or, here, time-frequency bins) where the observed signal differs significantly between experimental groups. This layer implements the **group contrast map** component of SPM:

**Step 1 — Group-mean scalogram in decibels.**
For each group $g \in \{\mathrm{HC}, \mathrm{MCI}\}$, compute the mean CWT power across all $N_g$ epochs:

$$\bar{P}_g(f, t) = \frac{1}{N_g} \sum_{i=1}^{N_g} \left|W_\psi[\tilde{e}_i](f, t)\right|^2$$

Convert to decibels to compress the dynamic range and work on a perceptually linear scale:

$$\overline{\mathrm{CWT}}_g(f, t)_{\mathrm{dB}} = 10 \log_{10}\bigl(\bar{P}_g(f, t) + \epsilon\bigr)$$

**Step 2 — Group contrast (Difference Map).**
The difference map is the MCI-minus-HC contrast:

$$\Delta(f, t) = \overline{\mathrm{CWT}}_{\mathrm{MCI}}(f, t)_{\mathrm{dB}} - \overline{\mathrm{CWT}}_{\mathrm{HC}}(f, t)_{\mathrm{dB}}$$

**Interpretation of $\Delta(f, t)$:**
- **Red regions** ($\Delta > 0$): MCI patients exhibit *higher* gaze error power at frequency $f$ and time $t$ post-stimulus. This indicates oscillatory instability or prolonged tracking failure specific to MCI.
- **Blue regions** ($\Delta < 0$): HC subjects exhibit higher power, indicating more vigorous corrective saccades or faster re-fixation.

The diverging colormap (`RdBu_r`) is centered at zero, with the white dashed vertical line marking $t = 0$ (stimulus onset). The diagram provides **clinical face validity** for the model before any ML training, demonstrating that the raw signal features are group-discriminant in physiologically plausible time-frequency regions.

> **Note on multiple comparisons:** The current implementation plots the raw mean difference without thresholding. For formal hypothesis testing (e.g., cluster-based permutation tests or FDR correction across time-frequency bins), a statistical thresholding step would be applied on top of $\Delta(f, t)$ — a natural extension for a full neuroimaging-style SPM analysis.

In [ ]:
# =========================================================================
# [Layer 5] XAI Visualizer: SPM-style Difference Maps
# Computes group-mean CWT power (dB) and MCI - HC difference map
# Red regions = frequencies/times where MCI patients show higher gaze error
# =========================================================================
class XAIVisualizer:
    def __init__(self, pipeline: EventLockedCWTPipeline):
        self.frequencies = pipeline.frequencies
        self.time_bins   = pipeline.target_time_bins
        self.pre_sec     = pipeline.pre_sec
        self.post_sec    = pipeline.post_sec

    def _compute_group_mean_db(self, data_store):
        group_powers = defaultdict(list)
        for group, subjects in data_store.items():
            for subject_id, tasks in subjects.items():
                for task, eyes in tasks.items():
                    for eye, tensors in eyes.items():
                        for tensor in tensors:  # [2, freq, time]
                            # Reconstruct power from 2-channel complex representation
                            power = tensor[0] ** 2 + tensor[1] ** 2
                            group_powers[group].append(power)
        return {
            group: 10 * np.log10(np.mean(np.array(powers), axis=0) + 1e-10)
            for group, powers in group_powers.items()
        }

    def plot_difference_map(self, data_store, save_path=None):
        group_mean_db = self._compute_group_mean_db(data_store)
        if 'HC' not in group_mean_db or 'MCI' not in group_mean_db:
            print("[!] HC와 MCI 데이터가 모두 필요합니다.")
            return

        diff_map  = group_mean_db['MCI'] - group_mean_db['HC']
        time_axis = np.linspace(-self.pre_sec, self.post_sec, self.time_bins)
        extent    = [time_axis[0], time_axis[-1], self.frequencies[0], self.frequencies[-1]]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        configs = [
            (group_mean_db['HC'],  'HC Mean CWT (dB)',          'viridis', None,   None),
            (group_mean_db['MCI'], 'MCI Mean CWT (dB)',         'viridis', None,   None),
            (diff_map,             'Difference: MCI \u2212 HC (dB)', 'RdBu_r',
             -np.abs(diff_map).max(), np.abs(diff_map).max()),
        ]
        for ax, (data, title, cmap, vmin, vmax) in zip(axes, configs):
            im = ax.imshow(data, aspect='auto', origin='lower', extent=extent,
                           cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axvline(x=0, color='white', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.set_title(title, fontsize=12)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Frequency (Hz)')
            cbar = plt.colorbar(im, ax=ax)
            cbar.set_label('dB' if 'Difference' not in title else '\u0394dB')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"[+] Difference Map 저장됨: {save_path}")
        plt.show()

---

## Execution

The cell below runs the full pipeline in sequence:

1. **ETL** — scan `../data/` recursively, extract event-locked CWT epochs from all HC and MCI saccade CSVs
2. **XAI** — generate and save the SPM-style group contrast map *before* training, providing a model-agnostic validation of the feature space
3. **Training** — subject-level stratified split → weighted CE loss → 20 epochs with cosine LR → save best checkpoint
4. **Inference demo** — load best checkpoint, run soft-voting ensemble on the first CSV found in the data directory

> **Hardware note:** With `device="cuda"`, training uses FP16 mixed precision on the Jetson AGX Orin's Tensor Cores. On CPU, the code falls back to FP32 automatically with AMP disabled.

In [ ]:
if __name__ == "__main__":
    DATA_DIR = Path("../data")

    pipeline_config = {
        "pre_stimulus_sec":  0.2,
        "post_stimulus_sec": 0.8,
        "min_freq":          1.0,
        "max_freq":          40.0,
        "freq_bins":         40,
        "target_time_bins":  100,
        "w_morlet":          5.0,
    }
    pipeline = EventLockedCWTPipeline(**pipeline_config)

    if not DATA_DIR.exists():
        print(f"[!] 데이터 경로가 존재하지 않습니다: {DATA_DIR}")
    else:
        print(f"[*] 데이터 로드 중: {DATA_DIR.resolve()}")
        pipeline.process_directory(DATA_DIR)

        if not pipeline.data_store:
            print("[!] 처리된 데이터가 없습니다. 경로 및 파일명을 확인하세요.")
        else:
            # --- XAI: Difference Maps (before training) ---
            visualizer = XAIVisualizer(pipeline)
            visualizer.plot_difference_map(pipeline.data_store, save_path="xai_difference_map.png")

            # --- Dataset ---
            dataset = VOG_CWT_Dataset(pipeline.data_store)
            n_hc  = (dataset.y == 0).sum().item()
            n_mci = (dataset.y == 1).sum().item()
            print(f"[*] 총 {len(dataset)}개 CWT 샘플 (HC: {n_hc}, MCI: {n_mci})")

            # --- Training ---
            model   = EdgeCWTClassifier(num_classes=2)
            trainer = ModelTrainer(model, device="cuda")
            trainer.train_model(dataset, epochs=20, batch_size=32)

            # --- Inference example ---
            engine = JetsonInferenceEngine('best_edge_cwt_model.pth', pipeline_config)
            sample_files = list(DATA_DIR.rglob('*.csv'))
            if sample_files:
                engine.infer_csv(sample_files[0])